Denne notebooken demonstrerer kode for hvordan man kan kjøre retrieval benchmarking mot vektordatabasen.

## Koble til vektordatabasen

In [ ]:
import os
from typing import cast

from pydantic import HttpUrl

from nks_kbs_analyse.auth import BrowserSessionAuthentication, BrowserType
from nks_kbs_analyse.retriever import NKSRetriever

In [ ]:
# merk: avhengig av hva du har som standard nettleser kan det hende at du kan må sette BROWSER og PROFILE_PATH som env-variabler
auth = BrowserSessionAuthentication(
    HttpUrl("https://nks-vdb.ansatt.dev.nav.no"),
    browser=cast(BrowserType, os.getenv("BROWSER")),
    profile_path=os.getenv("PROFILE_PATH"),
)
nks_vdb_retriever = NKSRetriever(auth=auth)

In [ ]:
# test for å se at connection til nks_vdb fungerer
docs = nks_vdb_retriever.invoke("Hva er samordning mellom dagpenger og sykepenger?")
docs

## Hent testcaser

Henter syntestiske testcaser generert til benchmarking kjørt høsten 2024. 

In [ ]:
from google.cloud import bigquery

bq_client = bigquery.Client("nks-aiautomatisering-prod-194a")

query = """
  select 
    s.id
    , s.knowledge_article_id
    , s.knowledge_column
    , s.question
    , a.ArticleType 
    , a.Title 
  from `nks-aiautomatisering-prod-194a.kunnskapsbase.syntetiske_sporsmal` s
  left join `nks-aiautomatisering-prod-194a.kunnskapsbase.kunnskapsartikler` a
  on s.knowledge_article_id = a.KnowledgeArticleId

  where DATETIME_TRUNC(created, DAY) = '2024-09-16'
  -- dropper de artiklene vi ikke får joinet lenger sfa. endringer i kunnskapsbasen
  and a.Title is not NULL
"""
alle_testcaser = list(bq_client.query(query).result())

In [ ]:
len(alle_testcaser)

## Utfør søk

In [ ]:
from nks_kbs_analyse.retrieval_eval import run_evaluations

Eksempel: config for hybridsøk der vektorsøk og tekstsøk vektes like tungt

In [ ]:
#
K = 5  # vil returnere resultater for hver k opp til satt K-verdi
FTS_WEIGHT = 1.0
SEMANTIC_WEIGHT = 1.0
WRITE_RESULTS_TO_BQ = False

In [ ]:
results = run_evaluations(
    nks_vdb_retriever,
    alle_testcaser,
    k=K,
    fts_weight=FTS_WEIGHT,
    semantic_weight=SEMANTIC_WEIGHT,
    write_results_to_bq=WRITE_RESULTS_TO_BQ,
    group_cols=[
        "fts_weight",
        "semantic_weight",
        "k",
    ],  # Default vil gruppere søket etter "knowledge_column" i tillegg. Her slår vi bare sammen alle casene
)

In [ ]:
results

Eksempel: config for å kun kjøre semantisk søk

In [ ]:
# ex: config for å kun vektlegge semantisk søk
K = 5  # vil returnere resultater for hver k opp til satt K-verdi
FTS_WEIGHT = 0.0
SEMANTIC_WEIGHT = 1.0
WRITE_RESULTS_TO_BQ = False

In [ ]:
results = run_evaluations(
    nks_vdb_retriever,
    alle_testcaser,
    k=K,
    fts_weight=FTS_WEIGHT,
    semantic_weight=SEMANTIC_WEIGHT,
    write_results_to_bq=WRITE_RESULTS_TO_BQ,
    group_cols=[
        "fts_weight",
        "semantic_weight",
        "k",
    ],  # Default vil gruppere søket etter "knowledge_column" i tillegg. Her slår vi bare sammen alle casene
)

In [ ]:
results